# Deep Learning for Conspiracy Detection
## Predicting conspiracy labels (yes/no/cant_tell) using Transformer Models

This notebook implements a deep learning approach using transformer models for:
1. **Conspiracy Label Prediction**: yes/no/cant_tell (multiclass classification)
2. **Model**: DistilBERT-based text classifier (using only text, no engineered features)
3. **Model Evaluation**: Training with validation split and evaluation on held-out validation set (100 samples)

**Note**: Unlike other notebooks that use engineered features, this approach uses only the raw text and learns representations directly from the text data.

## Environment Setup

This notebook requires **NumPy 2.x** and compatible packages. The virtual environment should be set up using the `recreate_venv.sh` script which installs all packages with NumPy 2.x compatibility.


In [1]:
# Verify environment setup and package versions
# Suppress tokenizers parallelism warning
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import and verify all packages
import numpy as np
import pandas as pd
import torch
from transformers import __version__ as transformers_version

print("=" * 60)
print("Environment Verification")
print("=" * 60)

# Check NumPy
print(f"✓ NumPy version: {np.__version__}")
if not np.__version__.startswith('2.'):
    print("⚠ WARNING: NumPy 2.x is required but version", np.__version__, "is installed")

# Check PyTorch
print(f"✓ PyTorch version: {torch.__version__}")
try:
    test_tensor = torch.tensor([1, 2, 3])
    test_numpy = test_tensor.numpy()
    print("✓ PyTorch-NumPy compatibility: OK")
except Exception as e:
    print(f"❌ PyTorch-NumPy compatibility issue: {e}")
    print("   This may indicate PyTorch was not compiled with NumPy 2.x support")
    print("   Please run: bash EDA-Rehydrated/recreate_venv.sh")

# Check Transformers
print(f"✓ Transformers version: {transformers_version}")

# Check CUDA
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
else:
    print("ℹ Training will use CPU")

print("=" * 60)
print("✓ Environment check complete")
print("=" * 60)

/home/husnain/semeval26_psycholinguistic_markers/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment Verification
✓ NumPy version: 2.1.2
✓ PyTorch version: 2.9.1+cpu
✓ PyTorch-NumPy compatibility: OK
✓ Transformers version: 4.57.3
✓ CUDA available: False
ℹ Training will use CPU
✓ Environment check complete


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure numpy is properly available for transformers
# This is a workaround for numpy 2.x compatibility issues
import sys
if 'numpy' not in sys.modules:
    import numpy
    sys.modules['numpy'] = numpy

# Deep learning libraries
import torch
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Libraries imported successfully!")
print("=" * 50)
print("Environment Information:")
print("=" * 50)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
else:
    print("⚠ Training will use CPU (slower than GPU)")
print("=" * 50)


Libraries imported successfully!
Environment Information:
✓ PyTorch version: 2.9.1+cpu
✓ NumPy version: 2.1.2
✓ CUDA available: False
⚠ Training will use CPU (slower than GPU)


## Load Data

We only need the `text` and `conspiracy` columns for deep learning. The transformer model will learn features directly from the text.


In [3]:
# Load base data - we only need text and labels
BASE = Path('../')
PROC = BASE / 'data_processed'

# Load base data from data_clean.csv (contains _id, text, conspiracy)
df = pd.read_csv(PROC / 'data_clean.csv')

print(f"Loaded {len(df)} documents")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['conspiracy'].value_counts())

# Keep only text and conspiracy columns (we don't need _id or other features)
df_model = df[['text', 'conspiracy']].copy()

# Remove any rows with missing text or labels
df_model = df_model.dropna(subset=['text', 'conspiracy'])
print(f"\nAfter removing missing values: {len(df_model)} documents")
print(f"\nLabel distribution after cleaning:")
print(df_model['conspiracy'].value_counts())

# Display sample texts
print("\n=== Sample Texts ===")
for label in ['yes', 'no', 'cant_tell']:
    sample = df_model[df_model['conspiracy'] == label].iloc[0]
    print(f"\nLabel: {label}")
    print(f"Text (first 200 chars): {sample['text'][:200]}...")


Loaded 3682 documents

Columns: ['_id', 'text', 'conspiracy']

Label distribution:
conspiracy
no           1708
yes          1303
cant_tell     671
Name: count, dtype: int64

After removing missing values: 3682 documents

Label distribution after cleaning:
conspiracy
no           1708
yes          1303
cant_tell     671
Name: count, dtype: int64

=== Sample Texts ===

Label: yes
Text (first 200 chars): A great article on what's taking place in Bolivia, referencing some similar US backed coups in the region as well as recounting some of Bolivia's history and western policy towards the country....

Label: no
Text (first 200 chars): Germany has upset other EU member states by securing a disproportionately large share of the bloc’s common pool of vaccines, according to a report. 
 Brussels  has ordered  roughly two billion doses i...

Label: cant_tell
Text (first 200 chars): Chris Lehto interviews Ashton Forbes about his deep dive investigation into the mysterious airline videos that recen

## Create Train/Test Split

Split the data into:
- **Training set**: Used for k-fold cross-validation (3582 samples)
- **Test set**: Held-out final evaluation set (100 samples) - same as other notebooks


In [4]:
# Load test set IDs (same as other notebooks - these are held out for final evaluation)
TEST_FILE = PROC / 'validation_set_ids.csv'  # Note: file is named validation_set_ids but we use it as test set

if TEST_FILE.exists():
    print("Loading existing test set...")
    test_ids_df = pd.read_csv(TEST_FILE)
    test_ids = set(test_ids_df['_id'].values)
    print(f"  Loaded {len(test_ids)} test IDs from {TEST_FILE.name}")
else:
    print("WARNING: Test set file not found. Creating new test set...")
    # Create test set using stratified sampling
    TEST_SIZE = 100
    target_fraction = TEST_SIZE / len(df)
    
    train_ids, test_ids_list = train_test_split(
        df['_id'].values,
        test_size=target_fraction,
        stratify=df['conspiracy'].values,
        random_state=42
    )
    test_ids = set(test_ids_list[:TEST_SIZE])
    
    # Save test set IDs
    test_ids_df = pd.DataFrame({'_id': list(test_ids)})
    test_ids_df.to_csv(TEST_FILE, index=False)
    print(f"  ✓ Saved test set to {TEST_FILE.name}")

# Split data using test IDs
# We need to merge back with _id to split properly
df_with_id = df[['_id', 'text', 'conspiracy']].dropna(subset=['text', 'conspiracy'])

# Test set: held out for final evaluation
test_df = df_with_id[df_with_id['_id'].isin(test_ids)].copy()
# Training set: will be used for k-fold cross-validation
train_df = df_with_id[~df_with_id['_id'].isin(test_ids)].copy()

print(f"\nTraining set size: {len(train_df)} samples (for k-fold CV)")
print(f"Training set class distribution:")
print(train_df['conspiracy'].value_counts().to_dict())

print(f"\nTest set size: {len(test_df)} samples (held out for final evaluation)")
print(f"Test set class distribution:")
print(test_df['conspiracy'].value_counts().to_dict())

# Remove _id column for model training
train_df = train_df[['text', 'conspiracy']].copy()
test_df = test_df[['text', 'conspiracy']].copy()


Loading existing test set...
  Loaded 100 test IDs from validation_set_ids.csv

Training set size: 3582 samples (for k-fold CV)
Training set class distribution:
{'no': 1662, 'yes': 1267, 'cant_tell': 653}

Test set size: 100 samples (held out for final evaluation)
Test set class distribution:
{'no': 46, 'yes': 36, 'cant_tell': 18}


## Prepare Data for Training

Set up a train/validation split on the training set. The test set will be used only for final evaluation.


In [5]:
# Model configuration
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512  # Maximum sequence length for tokenization
# Use a single train/validation split instead of k-fold CV
# This allows using more data for training
VALIDATION_SPLIT = 0.2  # 20% for validation, 80% for training

# Encode labels (fit on training set only)
label_encoder = LabelEncoder()
label_encoder.fit(train_df['conspiracy'])

# Create label mappings
label_to_id = {label: int(id) for label, id in zip(label_encoder.classes_, range(len(label_encoder.classes_)))}
id_to_label = {int(id): label for label, id in label_to_id.items()}

print(f"Label mappings:")
print(f"  Label to ID: {label_to_id}")
print(f"  ID to Label: {id_to_label}")
print(f"  Number of classes: {len(label_to_id)}")

# Prepare training data for k-fold CV
train_texts = train_df['text'].tolist()
train_labels = label_encoder.transform(train_df['conspiracy']).tolist()

# Prepare test set (will be used only at the end)
test_texts = test_df['text'].tolist()
test_labels = label_encoder.transform(test_df['conspiracy']).tolist()

print(f"\nTraining set: {len(train_texts)} samples (will be split into train/validation)")
print(f"Test set: {len(test_texts)} samples (held out for final evaluation)")

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

print("\n✓ Data preparation complete!")
print("Tokenization will be done per fold to save memory")


Label mappings:
  Label to ID: {'cant_tell': 0, 'no': 1, 'yes': 2}
  ID to Label: {0: 'cant_tell', 1: 'no', 2: 'yes'}
  Number of classes: 3

Training set: 3582 samples (will be split into train/validation)
Test set: 100 samples (held out for final evaluation)



✓ Data preparation complete!
Tokenization will be done per fold to save memory


## Train/Validation Split Setup

Set up a stratified train/validation split on the training set.


In [6]:
# Set up train/validation split
from sklearn.model_selection import train_test_split

# Create stratified train/validation split
train_indices, val_indices = train_test_split(
    range(len(train_texts)),
    test_size=VALIDATION_SPLIT,
    stratify=train_labels,
    random_state=42
)

print(f"Train/Validation Split Setup")
print("=" * 60)
print(f"Training samples: {len(train_indices)} ({100*(1-VALIDATION_SPLIT):.1f}%)")
print(f"Validation samples: {len(val_indices)} ({100*VALIDATION_SPLIT:.1f}%)")

# Show class distribution in validation set
val_labels_split = [train_labels[i] for i in val_indices]
val_label_counts = pd.Series(val_labels_split).value_counts().sort_index()
print(f"\nValidation set class distribution:")
print(dict(zip([id_to_label[i] for i in val_label_counts.index], val_label_counts.values)))

# Show class distribution in training set
train_labels_split = [train_labels[i] for i in train_indices]
train_label_counts = pd.Series(train_labels_split).value_counts().sort_index()
print(f"\nTraining set class distribution:")
print(dict(zip([id_to_label[i] for i in train_label_counts.index], train_label_counts.values)))
print("=" * 60)

# Training hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
num_labels = len(label_to_id)

print(f"\nTraining Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Number of classes: {num_labels}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Validation split: {VALIDATION_SPLIT*100:.0f}%")


Train/Validation Split Setup
Training samples: 2865 (80.0%)
Validation samples: 717 (20.0%)

Validation set class distribution:
{'cant_tell': np.int64(131), 'no': np.int64(333), 'yes': np.int64(253)}

Training set class distribution:
{'cant_tell': np.int64(522), 'no': np.int64(1329), 'yes': np.int64(1014)}

Training Configuration:
  Model: distilbert-base-uncased
  Number of classes: 3
  Batch size: 16
  Learning rate: 2e-05
  Epochs: 5
  Weight decay: 0.01
  Validation split: 20%


In [7]:
# Workaround for numpy 2.x compatibility with transformers
# This ensures transformers can properly detect and use numpy
import transformers.trainer_pt_utils as trainer_utils
import torch

# Patch the numpy conversion to work with numpy 2.x
_original_nested_numpify = trainer_utils.nested_numpify

def patched_nested_numpify(tensors):
    """Patched version that works with numpy 2.x"""
    if isinstance(tensors, (list, tuple)):
        return type(tensors)(patched_nested_numpify(t) for t in tensors)
    elif isinstance(tensors, dict):
        return {k: patched_nested_numpify(v) for k, v in tensors.items()}
    elif isinstance(tensors, torch.Tensor):
        # Handle bfloat16 as in original
        if tensors.dtype == torch.bfloat16:
            tensors = tensors.to(torch.float32)
        # Convert to numpy - this should work with numpy 2.x
        return tensors.detach().cpu().numpy()
    else:
        return tensors

# Apply the patch
trainer_utils.nested_numpify = patched_nested_numpify
print("✓ Applied numpy 2.x compatibility patch for transformers")


✓ Applied numpy 2.x compatibility patch for transformers


## Define Metrics Function

Metrics function for evaluation during cross-validation.


In [8]:
# Define metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Convert to numpy arrays if they're not already
    if hasattr(predictions, 'numpy'):
        predictions = predictions.numpy()
    elif not isinstance(predictions, np.ndarray):
        predictions = np.array(predictions)
    
    if hasattr(labels, 'numpy'):
        labels = labels.numpy()
    elif not isinstance(labels, np.ndarray):
        labels = np.array(labels)
    
    predictions = np.argmax(predictions, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

print("✓ Metrics function defined")


✓ Metrics function defined


## Model Training

Train a single model using the train/validation split.


In [ ]:
# Model Training
print("=" * 60)
print("Starting Model Training")
print("=" * 60)

# Get train/validation data
train_split_texts = [train_texts[i] for i in train_indices]
train_split_labels = [train_labels[i] for i in train_indices]
val_split_texts = [train_texts[i] for i in val_indices]
val_split_labels = [train_labels[i] for i in val_indices]

print(f"Training samples: {len(train_split_texts)}")
print(f"Validation samples: {len(val_split_texts)}")

# Create datasets
train_dataset = Dataset.from_dict({
    'text': train_split_texts,
    'labels': train_split_labels
})
val_dataset = Dataset.from_dict({
    'text': val_split_texts,
    'labels': val_split_labels
})

# Tokenize
print("\nTokenizing data...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Load model
print(f"\nLoading {MODEL_NAME} with {num_labels} labels...")
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)

# Training arguments
training_args = TrainingArguments(
    output_dir='./distilbert-conspiracy-classification',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available(),
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train
print(f"\nTraining model for {NUM_EPOCHS} epochs...")
try:
    train_result = trainer.train()
    
    # Evaluate on validation set
    print("\nEvaluating on validation set...")
    eval_result = trainer.evaluate()
    
    print(f"\n{'='*60}")
    print("Training Complete!")
    print(f"{'='*60}")
    print(f"\nTraining Loss: {train_result.training_loss:.4f}")
    print(f"\nValidation Set Performance:")
    print(f"  Accuracy: {eval_result.get('eval_accuracy', 0):.4f}")
    print(f"  F1 (macro): {eval_result.get('eval_f1_macro', 0):.4f}")
    print(f"  F1 (weighted): {eval_result.get('eval_f1_weighted', 0):.4f}")
    
except Exception as e:
    print(f"❌ Training failed: {e}")
    import traceback
    traceback.print_exc()
    raise


Starting Model Training
Training samples: 2865
Validation samples: 717

Tokenizing data...


Map:   0%|          | 0/2865 [00:00<?, ? examples/s]

Map: 100%|██████████| 717/717 [00:00<00:00, 2708.02 examples/s]



Loading distilbert-base-uncased with 3 labels...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training model for 5 epochs...


Epoch,Training Loss,Validation Loss


## Final Evaluation on Test Set

Evaluate the best model from cross-validation on the held-out test set (100 samples).


In [ ]:
# Prepare test set for evaluation
print("Preparing test set for final evaluation...")
test_dataset = Dataset.from_dict({
    'text': test_texts,
    'labels': test_labels
})

# Tokenize test set
test_dataset = test_dataset.map(tokenize_function, batched=True)
print(f"✓ Test set prepared: {len(test_dataset)} samples")

# Load the best model from training
print("\nLoading best model from training...")
import glob
checkpoint_dirs = glob.glob('./distilbert-conspiracy-classification/checkpoint-*')
if checkpoint_dirs:
    # Get the latest checkpoint
    latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split('-')[-1]))
    best_model = DistilBertForSequenceClassification.from_pretrained(latest_checkpoint)
    print(f"✓ Loaded model from {latest_checkpoint}")
else:
    # Use the current model if no checkpoint found
    print("No checkpoint found, using current model")
    best_model = model

# Evaluate on test set
print("\n" + "=" * 60)
print("Evaluating on Test Set (100 held-out samples)")
print("=" * 60)

test_trainer = Trainer(
    model=best_model,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

try:
    test_results = test_trainer.evaluate(test_dataset)
    
    print("\n=== Test Set Performance ===")
    print("=" * 60)
    for key, value in test_results.items():
        if 'loss' not in key:  # Skip loss metrics for cleaner output
            print(f"  {key}: {value:.4f}")
    print("=" * 60)
except Exception as e:
    print(f"❌ Evaluation failed with error: {e}")
    import traceback
    traceback.print_exc()
    raise


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Preparing test set for final evaluation...


Map: 100%|██████████| 100/100 [00:00<00:00, 1457.21 examples/s]

✓ Test set prepared: 100 samples


NameError: name 'cv_df' is not defined

In [ ]:
# Get predictions on test set
print("\nGenerating predictions on test set...")
predictions = test_trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)
true_labels = test_dataset['labels']

# Convert to label names
y_true_labels = [id_to_label[label] for label in true_labels]
y_pred_labels = [id_to_label[label] for label in predicted_classes]

print("\n=== Test Set Predictions ===")
print(f"\nTrue label distribution:")
print(pd.Series(y_true_labels).value_counts().to_dict())

print(f"\nPredicted label distribution:")
print(pd.Series(y_pred_labels).value_counts().to_dict())

# Calculate metrics
test_accuracy = accuracy_score(true_labels, predicted_classes)
test_f1_macro = f1_score(true_labels, predicted_classes, average='macro')
test_f1_weighted = f1_score(true_labels, predicted_classes, average='weighted')

print(f"\n=== Test Set Performance (Detailed) ===")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1 (macro): {test_f1_macro:.4f}")
print(f"F1 (weighted): {test_f1_weighted:.4f}")

# Detailed classification report
print(f"\n=== Detailed Classification Report (Test Set) ===")
print(classification_report(y_true_labels, y_pred_labels, target_names=label_encoder.classes_))



Generating predictions on test set...


NameError: name 'test_trainer' is not defined

In [ ]:
# Confusion matrix for test set
cm = confusion_matrix(true_labels, predicted_classes)
print(f"\n=== Confusion Matrix (Test Set) ===")
print("Rows = True labels, Columns = Predicted labels")
print(f"Label order: {list(label_encoder.classes_)}")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set (DistilBERT, K-Fold CV)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

# Save figure
figures_dir = BASE / 'figures'
figures_dir.mkdir(exist_ok=True)
plt.savefig(figures_dir / 'test_confusion_matrix_deeplearning_kfold.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved confusion matrix to figures/test_confusion_matrix_deeplearning_kfold.png")


## Sample Test Set Predictions


In [ ]:
# Compare predictions with true labels for some examples
print("\n=== Sample Test Set Predictions ===")
sample_indices = np.random.choice(len(test_df), size=min(10, len(test_df)), replace=False)

results_df = pd.DataFrame({
    'true_label': [y_true_labels[i] for i in sample_indices],
    'predicted_label': [y_pred_labels[i] for i in sample_indices],
    'text_preview': [test_df.iloc[i]['text'][:100] + '...' for i in sample_indices],
    'correct': [y_true_labels[i] == y_pred_labels[i] for i in sample_indices]
})

print(results_df.to_string(index=False))

print(f"\n✓ Test set evaluation complete!")
print(f"  Total test samples: {len(test_df)}")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1 (macro): {test_f1_macro:.4f}")
print(f"  F1 (weighted): {test_f1_weighted:.4f}")
print(f"  Correct predictions: {(np.array(y_true_labels) == np.array(y_pred_labels)).sum()}/{len(y_true_labels)}")


## Summary

This deep learning approach:
- **Uses only text and labels**: No engineered features needed - the transformer model learns representations directly from text
- **Model**: DistilBERT (lightweight BERT variant) for efficient training
- **Evaluation Strategy**:
  - **Train/Validation Split**: 80% training, 20% validation (stratified split)
  - **Final Test Set**: 100 held-out samples used only for final evaluation
  - **Metrics**: Accuracy, F1-macro, F1-weighted reported for validation and test set
  - **Training**: 5 epochs with early stopping based on validation performance
- **Advantages**: 
  - Can capture complex semantic patterns in text
  - No feature engineering required
  - Transfer learning from pre-trained language model
  - Robust evaluation through k-fold CV
- **Comparison with feature-based models**: Deep learning models often perform better on text classification tasks, especially when there's sufficient training data

**Note**: For production use, you might want to:
- Experiment with different transformer models (BERT, RoBERTa, etc.)
- Tune hyperparameters (learning rate, batch size, epochs)
- Use ensemble predictions from all k-fold models
- Apply class weighting if there's class imbalance
